# Real-Time Fraud Detection: Exploratory Data Analysis (EDA)

This notebook analyzes the synthetic transaction dataset generated by `fraudgnn.data.simulator`.

### Objectives
1. **Class Balance**: Inspect overall fraud prevalence and breakdown across the 5 financial fraud typologies.
2. **Amount Distributions**: Compare spending amounts across legitimate transactions, micro-probing, and ATO spikes.
3. **Graph Degree Distributions**: Analyze user, merchant, and device degree distributions, examining benign vs. fraudulent graph connectivity.
4. **Temporal Patterns**: Verify circadian diurnal rhythms and transaction volume over the 90-day simulation window.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. Load Data

In [ ]:
data_path = Path("../data/raw/transactions.parquet")
if not data_path.exists():
    data_path = Path("data/raw/transactions.parquet")

df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} transactions.")
print(f"Time span: {df['timestamp'].min()} to {df['timestamp'].max()}")
print("\nDataFrame Schema:")
df.info()

In [ ]:
df.head()

## 2. Class Balance & Fraud Typologies

In [ ]:
class_counts = df["is_fraud"].value_counts()
fraud_rate = df["is_fraud"].mean() * 100

print(f"Legitimate Transactions : {class_counts.get(0, 0):,} ({100 - fraud_rate:.2f}%)")
print(f"Fraud Transactions      : {class_counts.get(1, 0):,} ({fraud_rate:.2f}%)")

typology_counts = df["fraud_type"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Binary Class Balance
ax1.bar(["Legitimate (0)", "Fraud (1)"], [class_counts.get(0, 0), class_counts.get(1, 0)], color=["#2b5c8f", "#d9534f"])
ax1.set_title("Binary Class Distribution", fontweight="bold")
ax1.set_ylabel("Number of Transactions")
ax1.set_yscale("log")

# Fraud Typology Breakdown (Fraud Only)
fraud_types = typology_counts.drop("legitimate", errors="ignore")
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#9b59b6", "#3498db"]
ax2.barh(fraud_types.index, fraud_types.values, color=colors[:len(fraud_types)])
ax2.set_title("Fraud Typology Distribution (Fraud Only)", fontweight="bold")
ax2.set_xlabel("Count")
plt.tight_layout()
plt.show()

## 3. Transaction Amount Distribution

We compare amounts between legitimate transactions and different fraud typologies on a logarithmic scale.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-amount distribution for Legit vs Fraud
legit_amt = df[df["is_fraud"] == 0]["amount"]
fraud_amt = df[df["is_fraud"] == 1]["amount"]

ax1.hist(np.log10(legit_amt), bins=50, alpha=0.6, label="Legitimate", color="#2b5c8f", density=True)
ax1.hist(np.log10(fraud_amt), bins=50, alpha=0.6, label="Fraud", color="#d9534f", density=True)
ax1.set_xlabel("log10(Amount in USD)")
ax1.set_ylabel("Density")
ax1.set_title("Amount Distribution: Legit vs Fraud (Log Scale)", fontweight="bold")
ax1.legend()

# Boxplot by fraud type
sample_df = df.sample(n=min(50000, len(df)), random_state=42)
types = [t for t in df["fraud_type"].unique() if t != "legitimate"]
box_data = [sample_df[sample_df["fraud_type"] == t]["amount"] for t in types]

ax2.boxplot(box_data, labels=types, vert=False, showfliers=False)
ax2.set_xlabel("Amount in USD")
ax2.set_title("Amount Spread by Fraud Typology (no outliers)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Graph Degree Distribution

In our heterogeneous transaction graph, nodes (users, merchants, devices, IPs) connect via transactions.
We inspect degree distributions to verify graph connectivity and assess whether benign noise creates shared-device connections.

In [ ]:
# Degree calculations
user_degrees = df.groupby("user_id").size()
device_degrees = df.groupby("device_id").size()
users_per_device = df.groupby("device_id")["user_id"].nunique()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Device Degree Distribution (log-log)
deg_counts = device_degrees.value_counts().sort_index()
ax1.loglog(deg_counts.index, deg_counts.values, marker="o", linestyle="none", color="#8e44ad", alpha=0.7)
ax1.set_xlabel("Degree (Transactions per Device)")
ax1.set_ylabel("Count of Devices")
ax1.set_title("Device Degree Distribution (Log-Log)", fontweight="bold")

# Users per Device (Benign Noise vs Shared)
user_dev_counts = users_per_device.value_counts().sort_index()
ax2.bar(user_dev_counts.index[:8], user_dev_counts.values[:8], color="#16a085")
ax2.set_xlabel("Distinct Users per Device")
ax2.set_ylabel("Number of Devices")
ax2.set_yscale("log")
ax2.set_title("Multi-User Devices (Benign Noise & Fraud Rings)", fontweight="bold")

plt.tight_layout()
plt.show()

shared_devices = (users_per_device > 1).sum()
print(f"Devices used by multiple users: {shared_devices:,} (confirms non-trivial graph structure)")

## 5. Temporal Circadian Analysis

Verify diurnal spending patterns and temporal distribution over 24-hour cycles.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
hourly_vol = df.groupby("hour").size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_vol.index, hourly_vol.values, marker="o", linewidth=2, color="#2980b9")
ax.set_xticks(range(24))
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Transaction Volume")
ax.set_title("Circadian Diurnal Rhythm: Volume by Hour of Day", fontweight="bold")
plt.tight_layout()
plt.show()